lấy 1000 ảnh ngẫu nhiên


In [ ]:
#lấy 1000 ảnh ngẫu nhiên
import random, shutil
from pathlib import Path

random.seed(42)

src_img = Path("/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/train/images")
src_lbl = Path("/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/train/labels")

all_imgs = list(src_img.glob("*.jpg")) + list(src_img.glob("*.png"))
sample = random.sample(all_imgs, min(1000, len(all_imgs)))

for dst_split in ["images/train_1k", "labels/train_1k"]:
    Path(f"/kaggle/working/fire_only/{dst_split}").mkdir(parents=True, exist_ok=True)

for img_path in sample:
    shutil.copy(img_path, f"/kaggle/working/fire_only/images/train_1k/{img_path.name}")
    lbl = src_lbl / (img_path.stem + ".txt")
    if lbl.exists():
        shutil.copy(lbl, f"/kaggle/working/fire_only/labels/train_1k/{lbl.name}")

print(f"Sampled {len(sample)} images")

chạy code so sánh


In [2]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00a 0:00:01


In [ ]:
from ultralytics import YOLO
import os, subprocess, torch

# Dọn dẹp
subprocess.run("pkill -f 'torch.distributed' 2>/dev/null", shell=True)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)
os.environ.pop("WORLD_SIZE", None)
torch.cuda.empty_cache()

model = YOLO("yolo11n.pt")

model.train(
    data        = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    epochs      = 50,
    imgsz       = 640,
    batch       = 64,
    device      = "0,1",
    workers     = 4,
    patience    = 15,
    save        = True,
    save_period = 10,
    plots       = True,
    project     = "/kaggle/working/fire_trial",
    name        = "run1k_default",
    exist_ok    = True,
)
import pandas as pd
from ultralytics import YOLO

RUN_DIR = "/kaggle/working/fire_trial/run1k_default"

df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = df.columns.str.strip()

best_epoch = df["metrics/mAP50(B)"].idxmax()
best = df.iloc[best_epoch]
last = df.iloc[-1]

print("="*55)
print("KẾT QUẢ TRAIN 1K ẢNH — DEFAULT PARAMS")
print("="*55)
print(f"  Epochs trained       : {len(df)}")
print(f"  Best epoch           : {int(best['epoch'])}")
print(f"  mAP@0.5   (best)     : {best['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5   (last)     : {last['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:95 (best)    : {best['metrics/mAP50-95(B)']:.4f}")
print(f"  Precision  (best)    : {best['metrics/precision(B)']:.4f}")
print(f"  Recall     (best)    : {best['metrics/recall(B)']:.4f}")
print(f"  Train box_loss (last): {last['train/box_loss']:.4f}")
print(f"  Val   box_loss (last): {last['val/box_loss']:.4f}")
print(f"  Train cls_loss (last): {last['train/cls_loss']:.4f}")
print(f"  Val   cls_loss (last): {last['val/cls_loss']:.4f}")

# Chẩn đoán sơ bộ
gap = last["val/box_loss"] - last["train/box_loss"]
map_best = best["metrics/mAP50(B)"]
print(f"\n  Val-Train gap        : {gap:.4f}", end=" → ")
if gap > 0.05:
    print(" Có dấu hiệu overfitting")
elif map_best < 0.5:
    print(" Underfitting — cần train thêm hoặc tăng model size")
elif map_best < 0.75:
    print(" Trung bình — có thể cải thiện params")
else:
    print(" Tốt")


In [ ]:

# Đánh giá trên val set
print("\n" + "="*55)
print("ĐÁNH GIÁ TRÊN VAL SET")
print("="*55)
best_model = YOLO(f"{RUN_DIR}/weights/best.pt")
val_metrics = best_model.val(
    data    = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    verbose = False,
)
print(f"  mAP@0.5    : {val_metrics.box.map50:.4f}")
print(f"  mAP@0.5:95 : {val_metrics.box.map:.4f}")
print(f"  Precision  : {val_metrics.box.mp:.4f}")
print(f"  Recall     : {val_metrics.box.mr:.4f}")

train full 9k

In [2]:
from ultralytics import YOLO
import os, subprocess, torch

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 — DỌN DẸP TRƯỚC KHI TRAIN
# Mục đích: tránh lỗi DDP do process cũ chưa tắt hẳn
# ═══════════════════════════════════════════════════════════
subprocess.run("pkill -f 'torch.distributed' 2>/dev/null", shell=True)
subprocess.run("pkill -f '_temp_' 2>/dev/null", shell=True)

# Xoá biến môi trường DDP còn sót từ session trước
for var in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(var, None)

# Giải phóng VRAM
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f" GPU sẵn sàng: {torch.cuda.device_count()} GPUs")
print(f"   GPU 0: {torch.cuda.get_device_name(0)}")
print(f"   GPU 1: {torch.cuda.get_device_name(1)}")


# ═══════════════════════════════════════════════════════════
# BƯỚC 2 — LOAD MODEL
# yolo11n.pt = pretrained trên COCO 80 classes
# Ta sẽ fine-tune lại chỉ với class 'fire'
# ═══════════════════════════════════════════════════════════
# model = YOLO("yolo11n.pt")
model = YOLO("/kaggle/working/fire_full/run_9k_v1/weights/last.pt")


# ═══════════════════════════════════════════════════════════
# BƯỚC 3 — TRAIN
# ═══════════════════════════════════════════════════════════
model.train(
    # ── Dataset ───────────────────────────────────────────
    data    = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    # File yaml chứa đường dẫn train/val/test và tên class
    # nc: 1, names: ['fire']

    # ── Số vòng lặp ───────────────────────────────────────
    epochs  = 150,
    # Tăng lên 150 vì lần test 1k ảnh best epoch = 47/50
    # → model chưa hội tụ hẳn, cần thêm epochs
    # patience=25 sẽ dừng sớm nếu không cải thiện

    patience = 25,
    # Early stopping: dừng nếu sau 25 epochs liên tiếp
    # mAP không tăng → tránh train thừa, tiết kiệm thời gian

    # ── Kích thước ảnh ────────────────────────────────────
    imgsz   = 640,
    # Ảnh resize về 640x640 trước khi đưa vào model
    # Cao hơn (ví dụ 1280) thì chính xác hơn nhưng chậm hơn
    # 640 là chuẩn tốt nhất cho yolo11n

    # ── Batch size ────────────────────────────────────────
    batch   = 64,
    # Tổng 64 ảnh/batch → mỗi GPU xử lý 32 ảnh
    # T4 16GB VRAM: với yolo11n + imgsz=640 thì 32/GPU là tối ưu
    # Nếu báo OOM (out of memory) thì giảm xuống batch=32

    # ── GPU ───────────────────────────────────────────────
    device  = "0,1",
    # Dùng cả 2 GPU T4 trên Kaggle
    # YOLO tự dùng DDP (Distributed Data Parallel)
    # → chia batch cho 2 GPU, tổng hợp gradient sau mỗi step

    # ── CPU workers ───────────────────────────────────────
    workers = 4,
    # Số CPU thread song song để load ảnh vào RAM
    # Kaggle có 4 CPU cores → đặt 4 là tối đa
    # Nếu train chậm bất thường thì thử giảm xuống 2

    # ══════════════════════════════════════════════════════
    # LEARNING RATE
    # ══════════════════════════════════════════════════════
    lr0     = 0.01,
    # Learning rate ban đầu
    # 0.01 là default YOLO, đang hoạt động tốt với kết quả 1k
    # Không thay đổi vì precision 0.907 cho thấy lr phù hợp

    lrf     = 0.005,
    # Learning rate cuối = lr0 × lrf = 0.01 × 0.005 = 0.00005
    # Giảm từ default 0.01 xuống 0.005
    # → lr decay sâu hơn ở cuối, giúp model hội tụ chặt hơn
    # → cải thiện mAP@0.5:95 (bbox khớp chặt hơn)

    momentum = 0.937,
    # Momentum của SGD optimizer — giữ nguyên default
    # Giúp gradient không bị dao động quá mạnh

    warmup_epochs = 3,
    # 3 epoch đầu lr tăng dần từ 0 → lr0
    # Tránh gradient explode khi mới bắt đầu train

    warmup_momentum = 0.8,
    # Momentum trong giai đoạn warmup thấp hơn (0.8 < 0.937)
    # Cho phép model "khởi động" linh hoạt hơn

    # ══════════════════════════════════════════════════════
    # REGULARIZATION — chống overfitting
    # ══════════════════════════════════════════════════════
    weight_decay = 0.001,
    # Tăng từ default 0.0005 lên 0.001
    # Lý do: val-train gap = 0.109 → có overfitting
    # weight_decay phạt các weight quá lớn → model tổng quát hơn

    # ══════════════════════════════════════════════════════
    # LOSS WEIGHTS — điều chỉnh mức độ ưu tiên từng loại loss
    # ══════════════════════════════════════════════════════
    box = 9.0,
    # Tăng từ default 7.5 lên 9.0
    # Lý do: mAP@0.5:95 = 0.60 còn thấp → bbox chưa khớp chặt
    # box loss cao hơn → model chú trọng hơn vào vị trí bbox
    # Phù hợp với lửa vì hình dạng bất thường, hay thay đổi

    cls = 0.5,
    # Classification loss — giữ nguyên default
    # Chỉ có 1 class (fire) nên cls loss ít quan trọng hơn

    dfl = 1.5,
    # Distribution Focal Loss — giữ nguyên default
    # Giúp dự đoán phân phối xác suất của bbox edge

    # ══════════════════════════════════════════════════════
    # COLOR AUGMENTATION — tăng độ đa dạng màu sắc
    # ══════════════════════════════════════════════════════
    hsv_h = 0.015,
    # Biến đổi Hue (màu sắc) ±1.5%
    # Nhỏ vì màu lửa (đỏ-cam) không nên thay đổi quá nhiều

    hsv_s = 0.8,
    # Biến đổi Saturation (độ bão hoà) ±80%
    # Tăng từ 0.7 lên 0.8
    # Lý do: lửa ban đêm vs ban ngày có độ bão hoà rất khác nhau

    hsv_v = 0.5,
    # Biến đổi Value (độ sáng) ±50%
    # Tăng từ 0.4 lên 0.5
    # Lý do: mô phỏng lửa trong điều kiện ánh sáng khác nhau
    # (phòng tối, ngoài trời, đêm, ngày)

    # ══════════════════════════════════════════════════════
    # GEOMETRIC AUGMENTATION
    # ══════════════════════════════════════════════════════
    fliplr = 0.5,
    # 50% xác suất lật ảnh theo chiều ngang
    # Hợp lý vì lửa có thể cháy từ trái hoặc phải

    flipud = 0.0,
    # KHÔNG lật dọc — lửa luôn cháy hướng lên trên
    # Lật dọc tạo ra ảnh phi thực tế → gây nhiễu

    degrees = 0.0,
    # KHÔNG xoay — lý do tương tự flipud
    # Lửa không bao giờ cháy nghiêng 90°

    translate = 0.1,
    # Dịch chuyển ảnh ±10% theo x và y
    # Giúp model detect lửa ở mọi vị trí trong frame

    scale = 0.5,
    # Scale ảnh trong khoảng [0.5, 1.5]
    # Giúp model detect lửa ở nhiều kích thước khác nhau
    # (đám cháy nhỏ xa vs đám cháy lớn gần)

    shear = 0.0,
    # KHÔNG shear — biến dạng này không thực tế với lửa

    perspective = 0.0,
    # KHÔNG perspective transform — không cần thiết

    # ══════════════════════════════════════════════════════
    # MIXING AUGMENTATION — kỹ thuật trộn ảnh nâng cao
    # ══════════════════════════════════════════════════════
    mosaic = 1.0,
    # Ghép 4 ảnh thành 1 — bật 100%
    # Rất hiệu quả cho detection: model thấy nhiều object/ảnh hơn
    # Đặc biệt tốt khi dataset có ảnh lửa đơn lẻ

    close_mosaic = 15,
    # Tắt mosaic ở 15 epoch cuối (epoch 135-150)
    # Tăng từ default 10 lên 15
    # Lý do: cho model "ổn định" với ảnh thực trước khi kết thúc

    mixup = 0.15,
    # Trộn 2 ảnh lại với nhau (alpha blending)
    # Tăng từ 0.0 lên 0.15
    # Lý do: giảm overfitting, model học feature tổng quát hơn
    # Đặc biệt giúp tăng recall (phát hiện lửa mờ/nhỏ)

    copy_paste = 0.1,
    # Copy object lửa từ ảnh này paste vào ảnh khác
    # Tăng từ 0.0 lên 0.1
    # Lý do: tăng số lượng instance lửa trong training
    # Giúp tăng recall — giảm bỏ sót lửa

    # ══════════════════════════════════════════════════════
    # LƯU & THEO DÕI
    # ══════════════════════════════════════════════════════
    save        = True,
    # Lưu best.pt và last.pt

    save_period = 10,
    # Lưu checkpoint mỗi 10 epoch (epoch10.pt, epoch20.pt...)
    # Quan trọng: nếu Kaggle sập có thể resume từ checkpoint gần nhất

    plots       = True,
    # Sinh tự động: results.png, PR_curve.png,
    # confusion_matrix.png, F1_curve.png
    # Dùng để phân tích và đưa vào báo cáo đồ án

    project     = "/kaggle/working/fire_full",
    name        = "run_9k_v1",
    # Kết quả lưu tại: /kaggle/working/fire_full/run_9k_v1/
    #   ├── weights/
    #   │   ├── best.pt   ← dùng cái này để inference
    #   │   └── last.pt   ← dùng để resume nếu sập
    #   ├── results.csv
    #   └── results.png
    
    exist_ok    = True,
    # Cho phép ghi đè vào folder cũ nếu đã tồn tại
    # Cần thiết khi resume sau khi sập
    resume=True
)


# ═══════════════════════════════════════════════════════════
# BƯỚC 4 — LƯU MODEL NGAY SAU KHI TRAIN XONG
# Kaggle có thể xoá /working sau khi session kết thúc
# → copy ra ngay để tránh mất
# ═══════════════════════════════════════════════════════════
import shutil

shutil.copy(
    "/kaggle/working/fire_full/run_9k_v1/weights/best.pt",
    "/kaggle/working/best_fire_model.pt"
)
print(" Đã lưu → /kaggle/working/best_fire_model.pt")


# ═══════════════════════════════════════════════════════════
# BƯỚC 5 — ĐÁNH GIÁ SAU KHI TRAIN
# ═══════════════════════════════════════════════════════════
import pandas as pd

RUN_DIR = "/kaggle/working/fire_full/run_9k_v1"
df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = df.columns.str.strip()

best_epoch = df["metrics/mAP50(B)"].idxmax()
best = df.iloc[best_epoch]
last = df.iloc[-1]

print("="*55)
print("KẾT QUẢ TRAIN FULL 9K ẢNH")
print("="*55)
print(f"  Epochs trained       : {len(df)}")
print(f"  Best epoch           : {int(best['epoch'])}")
print(f"  mAP@0.5   (best)     : {best['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:95 (best)    : {best['metrics/mAP50-95(B)']:.4f}")
print(f"  Precision  (best)    : {best['metrics/precision(B)']:.4f}")
print(f"  Recall     (best)    : {best['metrics/recall(B)']:.4f}")
print(f"  Train box_loss (last): {last['train/box_loss']:.4f}")
print(f"  Val   box_loss (last): {last['val/box_loss']:.4f}")

gap = last["val/box_loss"] - last["train/box_loss"]
print(f"  Val-Train gap        : {gap:.4f}", end=" → ")
if gap > 0.05:
    print(" Overfitting")
elif best["metrics/mAP50(B)"] < 0.75:
    print(" Underfitting")
else:
    print(" Ổn định")

# Đánh giá trên test set
print("\n" + "="*55)
print("ĐÁNH GIÁ TRÊN TEST SET")
print("="*55)
best_model = YOLO(f"{RUN_DIR}/weights/best.pt")
test_metrics = best_model.val(
    data    = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    split   = "test",
    verbose = False,
)
print(f"  mAP@0.5    : {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:95 : {test_metrics.box.map:.4f}")
print(f"  Precision  : {test_metrics.box.mp:.4f}")
print(f"  Recall     : {test_metrics.box.mr:.4f}")

✅ GPU sẵn sàng: 2 GPUs
   GPU 0: Tesla T4
   GPU 1: Tesla T4
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0

train lần 2

In [3]:
from ultralytics import YOLO
import os, subprocess, torch

subprocess.run("pkill -f 'torch.distributed' 2>/dev/null", shell=True)
for var in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(var, None)
torch.cuda.empty_cache()

model = YOLO("yolo11n.pt")

model.train(
    data    = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    epochs  = 120,
    # Giảm từ 150 xuống 120
    # Lý do: best epoch = 107, train thêm sau đó chỉ gây overfitting

    patience = 20,

    imgsz   = 640,
    batch   = 64,
    device  = "0,1",
    workers = 4,

    # ── Learning rate ──────────────────────────────────────
    lr0     = 0.008,
    # Giảm nhẹ từ 0.01 → 0.008
    # Lý do: model đã có base tốt, lr nhỏ hơn giúp
    # converge ổn định hơn, giảm dao động cuối quá trình

    lrf     = 0.005,
    momentum     = 0.937,
    warmup_epochs= 3,

    # ── Regularization — tăng mạnh để giảm gap 0.193 ──────
    weight_decay = 0.002,
    # Tăng từ 0.001 lên 0.002
    # Lý do: gap 0.193 quá cao → cần phạt weight nặng hơn

    # ── Loss weights ───────────────────────────────────────
    box = 9.0,
    # Giữ nguyên — đã cải thiện mAP@0.5:95 lên 0.615
    cls = 0.5,
    dfl = 1.5,

    # ── Color augmentation ─────────────────────────────────
    hsv_h = 0.015,
    hsv_s = 0.8,
    hsv_v = 0.5,

    # ── Geometric augmentation ─────────────────────────────
    fliplr    = 0.5,
    flipud    = 0.0,
    degrees   = 0.0,
    translate = 0.1,
    scale     = 0.5,
    shear     = 0.0,

    # ── Mixing augmentation — tăng để chống overfitting ────
    mosaic      = 1.0,
    close_mosaic= 15,

    mixup       = 0.25,
    # Tăng từ 0.15 lên 0.25
    # Lý do: mixup là kỹ thuật regularization mạnh nhất
    # trong augmentation — tăng lên để giảm gap val-train

    copy_paste  = 0.2,
    # Tăng từ 0.1 lên 0.2
    # Lý do: tăng thêm diversity của dataset
    # giúp model tổng quát hoá tốt hơn → giảm overfitting

    # ── Lưu & theo dõi ─────────────────────────────────────
    save        = True,
    save_period = 10,
    plots       = True,
    project     = "/kaggle/working/fire_full",
    name        = "run_9k_v2",
    exist_ok    = True,
)

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.005, mask_ratio=4, max_det=300, mixup=0.25, mode=tr

In [4]:
import pandas as pd
from ultralytics import YOLO

RUN_DIR = "/kaggle/working/fire_full/run_9k_v2"
df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = df.columns.str.strip()

best_epoch = df["metrics/mAP50(B)"].idxmax()
best = df.iloc[best_epoch]
last = df.iloc[-1]

print("="*55)
print("KẾT QUẢ TRAIN 9K — V2")
print("="*55)
print(f"  Epochs trained       : {len(df)}")
print(f"  Best epoch           : {int(best['epoch'])}")
print(f"  mAP@0.5   (best)     : {best['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:95 (best)    : {best['metrics/mAP50-95(B)']:.4f}")
print(f"  Precision  (best)    : {best['metrics/precision(B)']:.4f}")
print(f"  Recall     (best)    : {best['metrics/recall(B)']:.4f}")
print(f"  Train box_loss (last): {last['train/box_loss']:.4f}")
print(f"  Val   box_loss (last): {last['val/box_loss']:.4f}")

gap = last["val/box_loss"] - last["train/box_loss"]
print(f"  Val-Train gap        : {gap:.4f}", end=" → ")
if gap > 0.15:
    print(" Vẫn overfitting — cần tăng regularization thêm")
elif gap > 0.08:
    print(" Overfitting nhẹ — chấp nhận được")
else:
    print(" Ổn định")

# So sánh với v1
print("\n" + "="*55)
print("SO SÁNH V1 vs V2")
print("="*55)
v1 = {"mAP50": 0.9098, "mAP5095": 0.6155, "Precision": 0.9282, "Recall": 0.8547, "gap": 0.1933}
v2_map50   = best["metrics/mAP50(B)"]
v2_map5095 = best["metrics/mAP50-95(B)"]
v2_prec    = best["metrics/precision(B)"]
v2_rec     = best["metrics/recall(B)"]

def arrow(new, old):
    diff = new - old
    return f"+{diff:.4f} ↑" if diff > 0 else f"{diff:.4f} ↓"

print(f"  mAP@0.5    : {v1['mAP50']:.4f} → {v2_map50:.4f}   {arrow(v2_map50, v1['mAP50'])}")
print(f"  mAP@0.5:95 : {v1['mAP5095']:.4f} → {v2_map5095:.4f}   {arrow(v2_map5095, v1['mAP5095'])}")
print(f"  Precision  : {v1['Precision']:.4f} → {v2_prec:.4f}   {arrow(v2_prec, v1['Precision'])}")
print(f"  Recall     : {v1['Recall']:.4f} → {v2_rec:.4f}   {arrow(v2_rec, v1['Recall'])}")
print(f"  Gap        : {v1['gap']:.4f} → {gap:.4f}   {arrow(-gap, -v1['gap'])}")

# Đánh giá test set
print("\n" + "="*55)
print("ĐÁNH GIÁ TRÊN TEST SET")
print("="*55)
best_model = YOLO(f"{RUN_DIR}/weights/best.pt")
test_metrics = best_model.val(
    data    = "/kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/data.yaml",
    split   = "test",
    verbose = False,
)
print(f"  mAP@0.5    : {test_metrics.box.map50:.4f}")
print(f"  mAP@0.5:95 : {test_metrics.box.map:.4f}")
print(f"  Precision  : {test_metrics.box.mp:.4f}")
print(f"  Recall     : {test_metrics.box.mr:.4f}")

import shutil
shutil.copy(f"{RUN_DIR}/weights/best.pt", "/kaggle/working/best_fire_model_v2.pt")
print("\n Đã lưu → /kaggle/working/best_fire_model_v2.pt")

KẾT QUẢ TRAIN 9K — V2
  Epochs trained       : 120
  Best epoch           : 105
  mAP@0.5   (best)     : 0.9022
  mAP@0.5:95 (best)    : 0.5985
  Precision  (best)    : 0.8974
  Recall     (best)    : 0.8609
  Train box_loss (last): 1.1172
  Val   box_loss (last): 1.4631
  Val-Train gap        : 0.3459 → ⚠️ Vẫn overfitting — cần tăng regularization thêm

SO SÁNH V1 vs V2
  mAP@0.5    : 0.9098 → 0.9022   -0.0076 ↓
  mAP@0.5:95 : 0.6155 → 0.5985   -0.0170 ↓
  Precision  : 0.9282 → 0.8974   -0.0308 ↓
  Recall     : 0.8547 → 0.8609   +0.0062 ↑
  Gap        : 0.1933 → 0.3459   -0.1526 ↓

ĐÁNH GIÁ TRÊN TEST SET
Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 4.3±0.8 ms, read: 21.1±9.7 MB/s, size: 34.4 KB)
val: Scanning /kaggle/input/datasets/daohiep/9k-fire-data/Fire-Smoke-Detection-Yolov11.v2-smoke-fire-detection-extra-augmentation.yolov11/test

In [6]:
import zipfile
import os
import shutil
from pathlib import Path

# ═══════════════════════════════════════════════════════════
# Đường dẫn các thư mục kết quả
# ═══════════════════════════════════════════════════════════
RUNS = {
    "run_1k_default" : "/kaggle/working/fire_trial/run1k_default",
    "run_9k_v1"      : "/kaggle/working/fire_full/run_9k_v1",
    "run_9k_v2"      : "/kaggle/working/fire_full/run_9k_v2",
    
}

# File cần lấy từ mỗi run
PLOTS = [
    "results.png",
    "PR_curve.png",
    "F1_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "labels.jpg",
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
]

ZIP_PATH = "/kaggle/working/fire_detection_results.zip"

# ═══════════════════════════════════════════════════════════
# Tạo file zip
# ═══════════════════════════════════════════════════════════
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:

    for run_name, run_dir in RUNS.items():
        run_path = Path(run_dir)

        if not run_path.exists():
            print(f"⚠️  Không tìm thấy: {run_dir} — bỏ qua")
            continue

        # 1. Ảnh đồ thị
        for plot in PLOTS:
            src = run_path / plot
            if src.exists():
                zf.write(src, f"{run_name}/plots/{plot}")
                print(f"  ✅ {run_name}/plots/{plot}")

        # 2. results.csv — quan trọng để vẽ biểu đồ trong báo cáo
        csv = run_path / "results.csv"
        if csv.exists():
            zf.write(csv, f"{run_name}/results.csv")
            print(f"  ✅ {run_name}/results.csv")

        # 3. best.pt — model tốt nhất của mỗi run
        best_pt = run_path / "weights" / "best.pt"
        if best_pt.exists():
            zf.write(best_pt, f"{run_name}/weights/best.pt")
            print(f"  ✅ {run_name}/weights/best.pt")

        # 4. args.yaml — lưu lại toàn bộ params đã dùng
        args = run_path / "args.yaml"
        if args.exists():
            zf.write(args, f"{run_name}/args.yaml")
            print(f"  ✅ {run_name}/args.yaml")

    # 5. Thêm best hyperparameters nếu có
    hp_yaml = Path("/kaggle/working/best_hyperparameters.yaml")
    if hp_yaml.exists():
        zf.write(hp_yaml, "best_hyperparameters.yaml")
        print(f"  ✅ best_hyperparameters.yaml")

    # 6. Thêm optuna results nếu có
    optuna_csv = Path("/kaggle/working/optuna_results.csv")
    if optuna_csv.exists():
        zf.write(optuna_csv, "optuna_results.csv")
        print(f"  ✅ optuna_results.csv")

# ═══════════════════════════════════════════════════════════
# Kiểm tra kết quả
# ═══════════════════════════════════════════════════════════
zip_size = Path(ZIP_PATH).stat().st_size / 1e6
print(f"\n{'='*50}")
print(f"✅ Đã tạo: {ZIP_PATH}")
print(f"📦 Kích thước: {zip_size:.1f} MB")
print(f"\nCấu trúc bên trong:")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        size = info.file_size / 1e6
        print(f"  {name:<55} ({size:.1f} MB)")

  ✅ run_1k_default/plots/results.png
  ✅ run_1k_default/plots/confusion_matrix.png
  ✅ run_1k_default/plots/confusion_matrix_normalized.png
  ✅ run_1k_default/plots/labels.jpg
  ✅ run_1k_default/plots/val_batch0_pred.jpg
  ✅ run_1k_default/plots/val_batch1_pred.jpg
  ✅ run_1k_default/results.csv
  ✅ run_1k_default/weights/best.pt
  ✅ run_1k_default/args.yaml
  ✅ run_9k_v1/plots/results.png
  ✅ run_9k_v1/plots/confusion_matrix.png
  ✅ run_9k_v1/plots/confusion_matrix_normalized.png
  ✅ run_9k_v1/plots/labels.jpg
  ✅ run_9k_v1/plots/val_batch0_pred.jpg
  ✅ run_9k_v1/plots/val_batch1_pred.jpg
  ✅ run_9k_v1/results.csv
  ✅ run_9k_v1/weights/best.pt
  ✅ run_9k_v1/args.yaml
  ✅ run_9k_v2/plots/results.png
  ✅ run_9k_v2/plots/confusion_matrix.png
  ✅ run_9k_v2/plots/confusion_matrix_normalized.png
  ✅ run_9k_v2/plots/labels.jpg
  ✅ run_9k_v2/plots/val_batch0_pred.jpg
  ✅ run_9k_v2/plots/val_batch1_pred.jpg
  ✅ run_9k_v2/results.csv
  ✅ run_9k_v2/weights/best.pt
  ✅ run_9k_v2/args.yaml

✅ Đã t